# GPT-2 Shakespeare

Fine-tunes GPT-2 on Shakespeare text (3 epochs) so it generates Shakespeare-style output.

## Fine-tune GPT-2 on the Shakespeare dataset

In [27]:
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from datasets import load_dataset, Dataset

from pathlib import Path
import urllib.request


In [3]:
def download_shakespeare_text():
    path = Path("datasets/shakespeare/shakespeare.txt")
    if not path.is_file():
        path.parent.mkdir(parents=True, exist_ok=True)
        url = "https://homl.info/shakespeare"
        urllib.request.urlretrieve(url, path)
    return path.read_text()

shakespeare_text = download_shakespeare_text()

In [ ]:
len(shakespeare_text)

1115394

In [5]:
type(shakespeare_text)

str

In [30]:
model_id='gpt2'
gpt2_tokenizer =  AutoTokenizer.from_pretrained(model_id)
gpt2_tokenizer.pad_token = gpt2_tokenizer.eos_token
gpt2_model = AutoModelForCausalLM.from_pretrained(model_id)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [20]:
tokenized_shapespeare_text = gpt2_tokenizer(shakespeare_text)

In [45]:
ids = tokenized_shapespeare_text["input_ids"]
blocks = [ids[i:i+512] for i in range(0, len(ids), 512)]
print(blocks[0])
lm_ds=Dataset.from_dict({"input_ids": blocks})

[5962, 22307, 25, 198, 8421, 356, 5120, 597, 2252, 11, 3285, 502, 2740, 13, 198, 198, 3237, 25, 198, 5248, 461, 11, 2740, 13, 198, 198, 5962, 22307, 25, 198, 1639, 389, 477, 12939, 2138, 284, 4656, 621, 284, 1145, 680, 30, 198, 198, 3237, 25, 198, 4965, 5634, 13, 12939, 13, 198, 198, 5962, 22307, 25, 198, 5962, 11, 345, 760, 327, 1872, 385, 1526, 28599, 318, 4039, 4472, 284, 262, 661, 13, 198, 198, 3237, 25, 198, 1135, 760, 470, 11, 356, 760, 470, 13, 198, 198, 5962, 22307, 25, 198, 5756, 514, 1494, 683, 11, 290, 356, 1183, 423, 11676, 379, 674, 898, 2756, 13, 198, 3792, 470, 257, 15593, 30, 198, 198, 3237, 25, 198, 2949, 517, 3375, 319, 470, 26, 1309, 340, 307, 1760, 25, 1497, 11, 1497, 0, 198, 198, 12211, 22307, 25, 198, 3198, 1573, 11, 922, 4290, 13, 198, 198, 5962, 22307, 25, 198, 1135, 389, 17830, 3595, 4290, 11, 262, 1458, 1173, 1547, 922, 13, 198, 2061, 4934, 969, 5036, 896, 319, 561, 26958, 514, 25, 611, 484, 198, 19188, 7800, 514, 475, 262, 48713, 414, 11, 981, 340, 547, 198, 

In [22]:
lm_ds

Dataset({
    features: ['input_ids'],
    num_rows: 661
})

In [23]:
print(len(ids), type(ids[0]))

338025 <class 'int'>


In [24]:
splits = lm_ds.train_test_split(test_size=0.1, seed=42)

In [25]:
splits

DatasetDict({
    train: Dataset({
        features: ['input_ids'],
        num_rows: 594
    })
    test: Dataset({
        features: ['input_ids'],
        num_rows: 67
    })
})

In [34]:
data_collator = DataCollatorForLanguageModeling(gpt2_tokenizer, mlm=False)

trraining_args = TrainingArguments(
    output_dir='gpt_shakesperean',
    num_train_epochs = 3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    bf16=True,
    weight_decay=0.1,
    learning_rate=2e-5,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy='epoch',
    load_best_model_at_end=True,
)


trainer = Trainer(
    model = gpt2_model,
    args = trraining_args,
    train_dataset = splits["train"],
    eval_dataset = splits["test"],
    processing_class=gpt2_tokenizer,
    data_collator = data_collator,
)
trainer.train()
trainer.push_to_hub()

Epoch,Training Loss,Validation Loss
1,No log,3.444354
2,No log,3.429614
3,3.527149,3.422992


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

CommitInfo(commit_url='https://huggingface.co/Cosminius/gpt_shakesperean/commit/55c6204a1ef3750f61fef90ee55d1abf6d65ccf9', commit_message='End of training', commit_description='', oid='55c6204a1ef3750f61fef90ee55d1abf6d65ccf9', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Cosminius/gpt_shakesperean', endpoint='https://huggingface.co', repo_type='model', repo_id='Cosminius/gpt_shakesperean'), pr_revision=None, pr_num=None)

In [42]:
prompt="Romeo you Romeo"
tokenized_prompt = gpt2_tokenizer(prompt, return_tensors='pt').to("cuda")
generated_text = gpt2_model.generate(**tokenized_prompt, max_new_tokens=100, do_sample=True, top_p=0.8)
gpt2_tokenizer.decode(generated_text)

["Romeo you Romeo!\n\nThe king will not permit this?\n\nKING ROMEO:\nWhy, my Lord.\n\nMARTINIUS:\nThy mother loves her child.\n\nKING ROMEO:\nThy son will marry me with me.\n\nMARTINIUS:\nWhat?\n\nKING ROMEO:\nWhat?\n\nMARTINIUS:\nAnd he'll come with me.\n\nKING ROMEO:"]